In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load dataset
df = pd.read_csv("Dataset .csv")

# Display first rows
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
# Select useful columns

data = df[[
    'Restaurant Name',
    'Cuisines',
    'City',
    'Price range',
    'Aggregate rating'
]]

data.head()

,Restaurant Name,Cuisines,City,Price range,Aggregate rating
0,Le Petit Souffle,"French, Japanese, Desserts",Makati City,3,4.8
1,Izakaya Kikufuji,Japanese,Makati City,3,4.5
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Mandaluyong City,4,4.4
3,Ooma,"Japanese, Sushi",Mandaluyong City,4,4.9
4,Sambo Kojin,"Japanese, Korean",Mandaluyong City,4,4.8


In [5]:
# Fill missing values

data['Cuisines'] = data['Cuisines'].fillna('Unknown')
data['City'] = data['City'].fillna('Unknown')

/tmp/ipykernel_3584/2864030711.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Cuisines'] = data['Cuisines'].fillna('Unknown')
/tmp/ipykernel_3584/2864030711.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['City'] = data['City'].fillna('Unknown')


In [6]:
# Combine features into single column

data['Features'] = (
    data['Cuisines'] + " " +
    data['City'] + " " +
    data['Price range'].astype(str)
)

data.head()

/tmp/ipykernel_3584/3974754825.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Features'] = (


,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Features
0,Le Petit Souffle,"French, Japanese, Desserts",Makati City,3,4.8,"French, Japanese, Desserts Makati City 3"
1,Izakaya Kikufuji,Japanese,Makati City,3,4.5,Japanese Makati City 3
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Mandaluyong City,4,4.4,"Seafood, Asian, Filipino, Indian Mandaluyong C..."
3,Ooma,"Japanese, Sushi",Mandaluyong City,4,4.9,"Japanese, Sushi Mandaluyong City 4"
4,Sambo Kojin,"Japanese, Korean",Mandaluyong City,4,4.8,"Japanese, Korean Mandaluyong City 4"


In [7]:
# Convert text data into matrix

cv = CountVectorizer()

feature_matrix = cv.fit_transform(data['Features'])

In [8]:
# Compute similarity scores

similarity = cosine_similarity(feature_matrix)

print(similarity)

[[1.         0.77459667 0.18257419 ... 0.         0.         0.        ]
 [0.77459667 1.         0.23570226 ... 0.         0.         0.        ]
 [0.18257419 0.23570226 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.28867513 0.35355339]
 [0.         0.         0.         ... 0.28867513 1.         0.81649658]
 [0.         0.         0.         ... 0.35355339 0.81649658 1.        ]]


In [12]:
def recommend_restaurant(name):

    index = data[data['Restaurant Name'] == name].index[0]

    scores = list(enumerate(similarity[index]))

    sorted_scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommended = sorted_scores[1:11]

    recommendations = []

    for item in recommended:

        restaurant_index = item[0]

        restaurant = data.iloc[restaurant_index]

        recommendations.append([
            restaurant['Restaurant Name'],
            restaurant['Cuisines'],
            restaurant['City'],
            restaurant['Aggregate rating']
        ])

    recommendation_df = pd.DataFrame(
        recommendations,
        columns=[
            'Restaurant',
            'Cuisine',
            'City',
            'Rating'
        ]
    )

    return recommendation_df

In [10]:
data['Restaurant Name'].head(20)

,Restaurant Name
0,Le Petit Souffle
1,Izakaya Kikufuji
2,Heat - Edsa Shangri-La
3,Ooma
4,Sambo Kojin
5,Din Tai Fung
6,Buffet 101
7,Vikings
8,Spiral - Sofitel Philippine Plaza Manila
9,Locavore


In [13]:
recommend_restaurant("Cafe Coffee Day")

,Restaurant,Cuisine,City,Rating
0,TcozY,Cafe,Faridabad,0.0
1,Cafe Bite,Cafe,Faridabad,3.2
2,Cafe Coffee Day,Cafe,Faridabad,3.3
3,The Binge Box Cafe,Cafe,Faridabad,3.4
4,Tmos Cafe Corner,Cafe,Faridabad,0.0
5,The Chaiwalas,Cafe,Faridabad,0.0
6,Tamasha In Tafree,Cafe,Faridabad,0.0
7,The Chocolate Room,"Cafe, Desserts",Faridabad,3.6
8,The Chai Cafe,"Cafe, Chinese",Faridabad,3.1
9,Funk House Cafe,"Cafe, Italian, Salad",Faridabad,2.6


# Insights

1. Restaurants with similar cuisines and price ranges were grouped together.

2. Content-based filtering successfully generated personalized recommendations.

3. Cuisine type played a major role in recommendation similarity.

4. Combining multiple restaurant features improved recommendation quality.

# Conclusion

This project developed a content-based restaurant recommendation system using restaurant features such as cuisines, city, and price range.

The recommendation engine used:
- Text vectorization
- Cosine similarity
- Feature combination

The system successfully recommended restaurants similar to user preferences.